In [2]:
!pip install torch transformers datasets pandas scikit-learn numpy tqdm

In [4]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
import torch
# Import AdamW from torch.optim instead of transformers
from torch.optim import AdamW



In [3]:
import torch
import numpy as np
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from torch import nn
from datasets import load_dataset
import pandas as pd
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Load and prepare data
def load_data():
    try:
        dataset = load_dataset("go_emotions")
        print("Dataset loaded successfully!")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Using sample data instead...")
        dataset = {
            'train': pd.DataFrame({
                'text': ["I love this!", "I hate this", "I'm okay with it"],
                'labels': [[1, 0, 0], [0, 0, 1], [0, 1, 0]]
            })
        }

    emotions = ['joy', 'neutral', 'anger']  # Example emotions
    texts = dataset['train']['text']
    # Pad labels to have consistent length
    labels = np.array([label[:len(emotions)] + [0] * (len(emotions) - len(label[:len(emotions)])) for label in dataset['train']['labels']])

    return texts, labels, emotions

texts, labels, emotions = load_data()

# 2. Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_data(texts, max_length=128):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

encoded_inputs = tokenize_data(texts.tolist() if hasattr(texts, 'tolist') else texts)

# 3. Create DataLoader
dataset = TensorDataset(
    encoded_inputs['input_ids'],
    encoded_inputs['attention_mask'],
    torch.FloatTensor(labels)
)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 4. Model initialization with proper handling
def initialize_model(num_labels):
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    # Initialize classifier weights properly
    nn.init.xavier_uniform_(model.classifier.weight)
    nn.init.zeros_(model.classifier.bias)

    return model.to(device)

model = initialize_model(len(emotions))

# 5. Training setup
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 3  # 3 epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)
criterion = nn.BCEWithLogitsLoss()

# 6. Training loop
def train_model():
    model.train()
    for epoch in range(3):
        total_loss = 0
        for batch in train_loader:
            batch = tuple(t.to(device) for t in batch)
            input_ids, attention_mask, labels = batch

            # optimizer.zero_grad()

            # outputs = model(
            #     input_ids,
            #     attention_mask=attention_mask,
            #     labels=labels
            # )

            # loss = outputs.loss
            # total_loss += loss.item()

            # loss.backward()
            # optimizer.step()
            # scheduler.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")

train_model()

# 7. Prediction function
def predict_emotion(text, threshold=0.5):
    model.eval()
    inputs = tokenize_data([text])

    with torch.no_grad():
        outputs = model(
            inputs['input_ids'].to(device),
            attention_mask=inputs['attention_mask'].to(device)
        )

    probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    return [(emotions[i], float(prob)) for i, prob in enumerate(probs) if prob > threshold]

# 8. Test the model
test_samples = [
    "I'm absolutely thrilled!",
    "This makes me furious",
    "It's just normal, nothing special"
]

for sample in test_samples:
    predictions = predict_emotion(sample)
    print(f"\nText: '{sample}'")
    for emotion, prob in predictions:
        print(f"- {emotion} (confidence: {prob:.2%})")

Dataset loaded successfully!


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 - Avg Loss: 0.0000
Epoch 2 - Avg Loss: 0.0000
Epoch 3 - Avg Loss: 0.0000

Text: 'I'm absolutely thrilled!'
- joy (confidence: 57.41%)
- neutral (confidence: 58.29%)

Text: 'This makes me furious'
- neutral (confidence: 59.29%)

Text: 'It's just normal, nothing special'
- joy (confidence: 58.43%)
- neutral (confidence: 65.52%)
